# PHASE 2 - Feature Engineering

This notebook creates calendar, lag, and rolling features from the hourly bike-demand data. It does not train a model or split the data.

Leakage rule: every lag and rolling feature is calculated from earlier rows only. In particular, rolling features use `target.shift(1)` so the target at the prediction timestamp is never included.

In [ ]:
# ========== BƯỚC 1: IMPORT VÀ LOAD DỮ LIỆU ==========
from pathlib import Path  # Thư viện làm việc với đường dẫn file
import pandas as pd  # Xử lý dữ liệu

# Hiển thị tất cả các cột khi in DataFrame
pd.set_option("display.max_columns", None)

# ========== TÌM KIẾM FILE DỮ LIỆU ==========
# Thử tìm file ở 2 vị trí có thể
data_candidates = [
    Path("data/raw/hour.csv"),  # Nếu chạy từ thư mục gốc
    Path("../data/raw/hour.csv"),  # Nếu chạy từ thư mục notebooks
]

# Tìm file đầu tiên tồn tại
data_path = next((path for path in data_candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Place the dataset at data/raw/hour.csv before running this notebook.")

# ========== ĐỌC VÀ TIỀN XỬ LÝ DỮ LIỆU ==========
# Đọc file CSV
df = pd.read_csv(data_path)

# Chuyển cột ngày từ string thành datetime
df["dteday"] = pd.to_datetime(df["dteday"], errors="raise")

# Tạo cột "timestamp" bằng cách kết hợp ngày + giờ
# Ví dụ: dteday="2011-01-01" + hr=2 → timestamp="2011-01-01 02:00:00"
df["timestamp"] = df["dteday"] + pd.to_timedelta(df["hr"], unit="h")

# Sắp xếp dữ liệu theo thời gian và đặt lại index
df = df.sort_values("timestamp").reset_index(drop=True)

# In thông tin cơ bản
print(f"Loaded: {data_path}")
print(f"Số hàng và cột: {df.shape}")
print(f"Khoảng thời gian: {df['timestamp'].min()} đến {df['timestamp'].max()}")
df.head()

Loaded: ..\data\raw\hour.csv
Raw shape: (17379, 18)
Timestamp range: 2011-01-01 00:00:00 to 2012-12-31 23:00:00


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt,timestamp
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16,2011-01-01 00:00:00
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40,2011-01-01 01:00:00
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32,2011-01-01 02:00:00
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13,2011-01-01 03:00:00
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1,2011-01-01 04:00:00


## Calendar features

In [ ]:
# ========== BƯỚC 2: TẠO CALENDAR FEATURES (NHỮNG ĐẶC ĐIỂM TỪ LỊCH) ==========
# Mục đích: Trích xuất thông tin thời gian từ timestamp để giúp model hiểu xu hướng theo giờ, ngày, tháng...

# Trích xuất từng thành phần từ timestamp
df["hour"] = df["timestamp"].dt.hour  # Giờ (0-23)
df["day"] = df["timestamp"].dt.day  # Ngày trong tháng (1-31)
df["month"] = df["timestamp"].dt.month  # Tháng (1-12)
df["year"] = df["timestamp"].dt.year  # Năm
df["day_of_week"] = df["timestamp"].dt.dayofweek  # Ngày trong tuần (0=Monday, 6=Sunday)
df["day_of_year"] = df["timestamp"].dt.dayofyear  # Ngày trong năm (1-365)

# ========== TẠO NHỮNG ĐẶC ĐIỂM NHÓM ==========
# Những đặc điểm này giúp model nhận biết các mô hình phức tạp

# Là ngày cuối tuần? (Saturday=5 hoặc Sunday=6) → 1 (có), 0 (không)
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# Là ngày làm việc bình thường? (đã có trong dữ liệu, chỉ convert sang int)
df["is_workingday"] = df["workingday"].astype(int)

# Là giờ cao điểm? (sáng: 7,8,9 hoặc chiều: 16,17,18,19) → 1 (có), 0 (không)
# Thường là những giờ đi làm/về nhà nên nhu cầu cao
df["rush_hour"] = df["hour"].isin([7, 8, 9, 16, 17, 18, 19]).astype(int)

# ========== DANH SÁCH CALENDAR FEATURES ==========
# Những đặc điểm lịch sẽ được sử dụng cho model
calendar_features = [
    "timestamp",  # Thời gian (để tham khảo)
    "hour",  # Giờ trong ngày
    "day",  # Ngày trong tháng
    "month",  # Tháng
    "year",  # Năm
    "day_of_week",  # Ngày trong tuần
    "day_of_year",  # Ngày trong năm
    "is_weekend",  # Là ngày cuối tuần?
    "is_workingday",  # Là ngày làm việc?
    "season",  # Mùa (đã có trong dữ liệu gốc)
    "rush_hour",  # Là giờ cao điểm?
]

# Hiển thị 5 hàng đầu tiên để kiểm tra
df[calendar_features].head()

,timestamp,hour,day,month,year,day_of_week,day_of_year,is_weekend,is_workingday,season,rush_hour
0,2011-01-01 00:00:00,0,1,1,2011,5,1,1,0,1,0
1,2011-01-01 01:00:00,1,1,1,2011,5,1,1,0,1,0
2,2011-01-01 02:00:00,2,1,1,2011,5,1,1,0,1,0
3,2011-01-01 03:00:00,3,1,1,2011,5,1,1,0,1,0
4,2011-01-01 04:00:00,4,1,1,2011,5,1,1,0,1,0


## Past-demand lag and rolling features

The dataset is ordered chronologically before `shift`. Therefore `lag_1`, `lag_2`, `lag_24`, and `lag_168` refer to earlier observations in the time-ordered data. The rolling windows are shifted first, which excludes the current target.

In [ ]:
# ========== BƯỚC 3: TẠO LAG FEATURES (ĐẶC ĐIỂM TỪ QUÁN KHỨ) ==========
# Mục đích: Sử dụng nhu cầu từ các thời điểm trước đó để dự báo hiện tại
# Lý do: Nhu cầu thường có tính chất lặp lại theo chu kỳ (giờ trước ảnh hưởng giờ hiện tại)

target = df["cnt"]  # Lấy cột "cnt" (số lượng xe thuê)

# Tạo lag features cho 4 khoảng thời gian khác nhau:
for lag in [1, 2, 24, 168]:
    # lag_1: Nhu cầu từ 1 giờ trước
    # lag_2: Nhu cầu từ 2 giờ trước
    # lag_24: Nhu cầu từ chính giờ này nhưng ngày hôm qua (1 ngày = 24 giờ)
    # lag_168: Nhu cầu từ chính giờ này nhưng tuần trước (1 tuần = 168 giờ)
    df[f"lag_{lag}"] = target.shift(lag)

# ========== BƯỚC 4: TẠO ROLLING FEATURES (TRUNG BÌNH QUÁN KHỨ) ==========
# Mục đích: Tính trung bình nhu cầu từ những ngày/tuần trước để capture xu hướng dài hạn

# Dịch chuyển target 1 bước để tránh leakage (không sử dụng nhu cầu hiện tại)
past_target = target.shift(1)

# Tính trung bình nhu cầu của 24 giờ trước đó (ngoại trừ giờ hiện tại)
# min_periods=24 nghĩa là chỉ tính khi có ít nhất 24 giá trị
df["rolling_mean_24"] = past_target.rolling(window=24, min_periods=24).mean()

# Tính trung bình nhu cầu của 168 giờ trước đó (1 tuần trước, ngoại trừ giờ hiện tại)
df["rolling_mean_168"] = past_target.rolling(window=168, min_periods=168).mean()

# ========== BƯỚC 5: TỔNG HỢP TẤT CẢ FEATURES ==========
# Liệt kê tất cả các loại feature sẽ dùng cho model

lag_features = ["lag_1", "lag_2", "lag_24", "lag_168"]  # Các lag features
rolling_features = ["rolling_mean_24", "rolling_mean_168"]  # Các rolling features
weather_features = ["temp", "atemp", "hum", "windspeed", "weathersit"]  # Thời tiết từ dữ liệu gốc
feature_columns = calendar_features[1:] + weather_features + lag_features + rolling_features
# Ghi chú: calendar_features[1:] bỏ đi "timestamp" vì timestamp không cần cho model

# Hiển thị 170 hàng đầu để xem lag/rolling features
df[["timestamp", "cnt"] + lag_features + rolling_features].head(170)

# ========== BƯỚC 6: KIỂM TRA KHÔNG CÓ LEAKAGE ==========
# Leakage = sử dụng thông tin từ tương lai để dự báo → NẠN trong ML
# Đoạn code này xác nhận rolling_mean_24 chỉ sử dụng nhu cầu từ quá khứ

# Tìm hàng đầu tiên có giá trị rolling_mean_24 (từ hàng 24)
rolling_start = int(df["rolling_mean_24"].first_valid_index())

# Lấy 24 giá trị "cnt" ngay trước đó
previous_targets = df["cnt"].iloc[rolling_start - 24:rolling_start]

# Tính trung bình của chúng
expected_rolling = previous_targets.mean()

# So sánh với giá trị rolling_mean_24 trong DataFrame
actual_rolling = df["rolling_mean_24"].iloc[rolling_start]

# Kiểm tra xem chúng có bằng nhau không (sai số < 1e-9)
assert abs(expected_rolling - actual_rolling) < 1e-9
print("✓ Kiểm tra leakage thành công: rolling_mean_24 chỉ sử dụng 24 nhu cầu trước đó")

# ========== BƯỚC 7: XÓA CÁC HÀNG CHƯA ĐỦ LỊCH SỬ ==========
# Những hàng đầu tiên chưa đủ dữ liệu lịch sử (vì lag_168 cần 168 hàng trước)

before_drop = len(df)  # Số hàng trước khi xóa

# Xóa các hàng có giá trị NaN (thiếu dữ liệu) trong lag/rolling features
# reset_index(drop=True) đặt lại index từ 0, 1, 2...
df_features = df.dropna(subset=lag_features + rolling_features).reset_index(drop=True)

print(f"Số hàng trước khi xóa hàng thiếu lịch sử: {before_drop}")
print(f"Số hàng sau feature engineering: {len(df_features)}")
print(f"Số giá trị NaN còn lại trong model features: {int(df_features[feature_columns].isna().sum().sum())}")

# Hiển thị 5 hàng đầu với tất cả features
df_features[["timestamp", "cnt"] + feature_columns].head()

# ========== BƯỚC 8: LƯU DỮ LIỆU ĐÃ XỬ LÝ ==========
# Tìm thư mục gốc project
project_root = Path(".") if Path("data/raw/hour.csv").exists() else Path("..")

# Tạo thư mục "data/processed" nếu chưa tồn tại
processed_dir = project_root / "data/processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# Tạo đường dẫn file output
output_path = processed_dir / "hour_features.csv"

# Lưu DataFrame vào file CSV
df_features.to_csv(output_path, index=False)

print(f"✓ Đã lưu dữ liệu xử lý vào: {output_path}")
print(f"Số lượng features: {len(feature_columns)}")

Leakage check passed: rolling_mean_24 uses the 24 preceding targets.
Rows before removing insufficient history: 17379
Rows after feature engineering: 17211
Remaining missing values in model features: 0
Saved processed features to: data\processed\hour_features.csv
Feature count: 21
